# SVM MODEL Experiments

**Objective:**

This notebook evaluates the performance of the SVM model using different strategies to handle class imbalance:

- Baseline
- Class-weight
- SMOTE
- SMOTEENN

**All experiments use:**
- A shared preprocessing pipeline
- Stratified K-Fold cross-validation
- A unified evaluation framework

Results are stored in a shared results table for comparison with other models.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent.parent / "bank-deposit-prediction"
sys.path.append(str(PROJECT_ROOT))

# Standard imports
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline

# Shared modules from Person 1
from src.shared import get_preprocessing_steps, get_cv
from src.evaluation import evaluate_model, save_results
from sklearn.model_selection import cross_validate


## Data Loading

In [2]:
data_path = PROJECT_ROOT / "data" / "raw" / "bank2.csv"

df = pd.read_csv(data_path, sep=';')

y = df['y'].map({'yes': 1, 'no': 0})
X = df.drop(columns=['y'])

## SVM PIPELINE SETUP 

In [3]:
cv_strategy = get_cv()

## Baseline SVM Model

In [5]:
all_results = []
preprocessor_steps = get_preprocessing_steps()

baseline_svm_pipeline = ImbPipeline(steps=preprocessor_steps + [
    ('classifier', SVC(
        kernel='rbf', 
        probability=True,      # Required for your Threshold Optimization task
        random_state=42, 
        class_weight=None      # 'None' makes this a true Baseline
    ))
])

# Evaluate model
result_baseline  = evaluate_model("SVM", "Baseline", baseline_svm_pipeline, X, y, cv_strategy)
all_results.append(result_baseline)
result_baseline

c:\Users\marwa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\marwa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\marwa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

{'Model': 'SVM',
 'Strategy': 'Baseline',
 'Accuracy': np.float64(0.883),
 'Precision': np.float64(0.0),
 'Recall': np.float64(0.0),
 'F1': np.float64(0.0),
 'ROC-AUC': np.float64(0.6339),
 'PR-AUC': np.float64(0.2661)}

## Class Weight (Imbalance Handling)

In [7]:
# Create the "Class-Weight" variation
svm_weighted = SVC(
    kernel='rbf', 
    class_weight='balanced', # <--- This tells SVM to care more about the 'Yes' group
    probability=True, 
    random_state=42
)

# Build the pipeline as usual
weighted_pipeline = ImbPipeline(steps=preprocessor_steps + [
    ('classifier', svm_weighted)
])

result_baseline  = evaluate_model("SVM", "Weighted", weighted_pipeline, X, y, cv_strategy)
all_results.append(result_baseline)
result_baseline

{'Model': 'SVM',
 'Strategy': 'Weighted',
 'Accuracy': np.float64(0.8255),
 'Precision': np.float64(0.2698),
 'Recall': np.float64(0.2938),
 'F1': np.float64(0.2798),
 'ROC-AUC': np.float64(0.7042),
 'PR-AUC': np.float64(0.2834)}

## SMOTE 

In [9]:
from imblearn.over_sampling import SMOTE
smote_svm_pipeline = ImbPipeline(steps=preprocessor_steps + [
    ('smote', SMOTE(random_state=42)), # Generates synthetic minority samples
    ('classifier', SVC(
        kernel='rbf', 
        probability=True, 
        random_state=42
    ))
])

result_smote = evaluate_model("SVM", "SMOTE", smote_svm_pipeline, X, y, cv_strategy)
all_results.append(result_smote)
result_smote

{'Model': 'SVM',
 'Strategy': 'SMOTE',
 'Accuracy': np.float64(0.8182),
 'Precision': np.float64(0.2639),
 'Recall': np.float64(0.3033),
 'F1': np.float64(0.2797),
 'ROC-AUC': np.float64(0.7021),
 'PR-AUC': np.float64(0.2801)}

## SMOTEEEN 

In [10]:
from imblearn.combine import SMOTEENN

smoteenn_svm_pipeline = ImbPipeline(steps=preprocessor_steps + [
    ('smoteenn', SMOTEENN(random_state=42)), # Oversamples AND cleans the boundary
    ('classifier', SVC(
        kernel='rbf', 
        probability=True, 
        random_state=42
    ))
])
result_smoteenn = evaluate_model("SVM", "SMOTEENN", smoteenn_svm_pipeline, X, y, cv_strategy)
all_results.append(result_smoteenn) 
result_smoteenn

{'Model': 'SVM',
 'Strategy': 'SMOTEENN',
 'Accuracy': np.float64(0.1157),
 'Precision': np.float64(0.1153),
 'Recall': np.float64(1.0),
 'F1': np.float64(0.2067),
 'ROC-AUC': np.float64(0.6819),
 'PR-AUC': np.float64(0.2297)}